In [1]:
import numpy as np
import pandas as pd
import os, json
import matplotlib.pyplot as plt

In [15]:
def get_result_json(dataset, encoder, model, root, verbose=False):
    log_dir = f'{root}/{dataset}/{encoder}/{model}'
    if verbose and not os.path.exists(log_dir):
        print(f'Log directory does not exist: {log_dir}')
        return None
    timestamps = os.listdir(log_dir)
    timestamps.sort()
    latest_timestamp = timestamps[-1]
    log_dir = f'{log_dir}/{latest_timestamp}'
    files = os.listdir(log_dir)
    num_seeds = len([f for f in files if f.startswith('seed')])
    log_files = [f for f in files if f.endswith('.json')]
    assert len(log_files) == 1, f"Expected exactly one log file, but found {len(log_files)} in {log_dir}"
    assert f"{num_seeds}_seeds" in log_files[0], f"Number of seeds {num_seeds} inconsistent with log file {log_files[0]}"
    if verbose:
        print(f'Found {len(log_files)} log files in {log_dir}')
    results = json.load(open(f'{log_dir}/{log_files[0]}', 'r'))
    return results

# After 30 Oct 2025

## Vanilla

### Collect results of non-attacked vanilla GNNs

In [ ]:
ROOT = "./tuned_results/vanilla"
files = os.listdir(ROOT)
lst_data = []
for file in files:
    if not file.endswith('.json'):
        continue
    path = os.path.join(ROOT, file)
    with open(path, 'r') as f:
        data = json.load(f)
    reformatted_data = {
        "dataset": file.split("__")[0],
        "backbone": file.split('__')[1].split('_')[0],
        "model": "vanilla",
        "attack_mode": "none",
        "lambda_dp": 0.0,
        "lambda_eo": 0.0,
        "AUC_mean": data["auc"]["mean"],
        "AUC_std": data["auc"]["std"],
        "F1_mean": data["f1"]["mean"],
        "F1_std": data["f1"]["std"],
        "DP_mean": data["dp"]["mean"],
        "DP_std": data["dp"]["std"],
        "EO_mean": data["eo"]["mean"],
        "EO_std": data["eo"]["std"],
    }
    lst_data.append(reformatted_data)
df_clean_tuned = pd.DataFrame(lst_data)
df_clean_tuned.shape

(25, 14)

In [ ]:
df_clean_tuned.to_csv('./vanilla_tuned_results.csv', index=False)

In [13]:
datasets = ['bail', 'pokec_z', 'pokec_n', 'nba', 'german']
encoders = ['gcn', 'gat', 'gin', 'sage', 'sgc']
models = ['vanilla']

In [41]:
path = "../logs/tuned_vanilla/no_attack"
verbose = False
lst_results = []
for dataset in datasets:
    for encoder in encoders:
        for model in models:
            results = get_result_json(dataset, encoder, model, path, verbose=verbose)
            if results:
                results['results']["dataset"] = dataset
                results['results']["encoder"] = encoder
                results['results']["model"] = model
                results['results']["attack_mode"] = "none"
                lst_results.append(results['results'])
df_clean = pd.DataFrame(lst_results).sort_values(by=["dataset", "encoder", "model"])

In [ ]:
path = "../logs/tuned_vanilla/nifa"
verbose = False
lst_results = []
for dataset in datasets:
    for encoder in encoders:
        for model in models:
            results = get_result_json(dataset, encoder, model, path, verbose=verbose)
            if results:
                results['results']["dataset"] = dataset
                results['results']["encoder"] = encoder
                results['results']["model"] = model
                results['results']["attack_mode"] = "degree"
                lst_results.append(results['results'])
df_attack = pd.DataFrame(lst_results).sort_values(by=["dataset", "encoder", "model"])

In [ ]:
assert df_clean[["dataset", "encoder", "model"]].equals(df_attack[["dataset", "encoder", "model"]])
df

In [56]:
numeric_cols = ["AUC_mean", "AUC_std", "ACC_mean", "ACC_std", "F1_mean", "F1_std", "DP_mean", "DP_std", "EO_mean", "EO_std"]
df_ratio = df_attack[numeric_cols] / df_clean[numeric_cols]
df_ratio[["dataset", "encoder", "model"]] = df_attack[["dataset", "encoder", "model"]]
df_ratio["attack_mode"] = "ratio"
df_vanilla = pd.concat([df_clean, df_attack, df_ratio], ignore_index=True)
df_vanilla["attack_mode"] = pd.Categorical(df_vanilla["attack_mode"], categories=["none", "degree", "ratio"], ordered=True)
df_vanilla = df_vanilla.sort_values(by=["dataset", "encoder", "model", "attack_mode"])
df_vanilla = df_vanilla[["dataset", "encoder", "model", "attack_mode"] + numeric_cols]
df_vanilla.to_csv('./vanilla_rerrun_before_and_after_nifa_degree.csv', index=False)

# After 21 Oct 2025

## Use vanilla-tuned HPs to train with DP/EO loss only

In [3]:
def aggregate_results(datasets, encoders, models, root, verbose=False):
    verbose = False
    Results = {}
    lst_results = []
    lst_scenarios = []
    exps = os.listdir(root)
    exps.sort()
    for exp in exps:
        Results[exp] = {}
        path = os.path.join(root, exp)
        for dataset in datasets:
            for encoder in encoders:
                for model in models:
                    try:
                        results = get_result_json(dataset, encoder, model, path, verbose=verbose)
                        Results[exp][f'{dataset}_{encoder}_{model}'] = results
                        if verbose:
                            print(f'{dataset}_{encoder}_{model}:')
                            for metrics in ['auc', 'f1', 'dp', 'eo']:
                                m = results['results'][f'{metrics.upper()}_mean']
                                s = results['results'][f'{metrics.upper()}_std']
                                print(f'{m:.2%} ± {s:.2%}')
                    except Exception as e:
                        if verbose:
                            print(f'Error processing {dataset}_{encoder}_{model}: {e}')
                    if verbose:
                        print()
        for k in Results[exp].keys():
            lst_scenarios.append((exp, k))
            lst_results.append(Results[exp][k]['results'])

    df = pd.DataFrame(
        lst_results,
        index=pd.MultiIndex.from_tuples(lst_scenarios, names=['experiment', 'scenario'])
    ).reset_index()

    df["dataset"] = df["scenario"].apply(lambda x: x.split('_')[0] if "pokec" not in x else "_".join(x.split('_')[:2]))
    df["backbone"] = df["scenario"].apply(lambda x: x.split('_')[1] if "pokec" not in x else x.split('_')[2])
    df["model"] = df["scenario"].apply(lambda x: "_".join(x.split('_')[-2:]))
    df["lambda_dp"] = df["experiment"].apply(lambda x: float(x.split('_')[1]))
    df["lambda_eo"] = df["experiment"].apply(lambda x: float(x.split('_')[-1]))
    df = df[['dataset', 'backbone', 'model',
         'lambda_dp', 'lambda_eo',
         'AUC_mean', 'AUC_std',
         'F1_mean', 'F1_std',
         'DP_mean', 'DP_std',
         'EO_mean', 'EO_std']]

    return df.sort_values(by=['dataset', 'backbone', 'model', 'lambda_dp', 'lambda_eo'])

In [4]:
Results = {}
root = '../logs/use_tuned_vanilla_hp_run_ea'
exps = os.listdir(root)
exps.sort()

In [7]:
df = aggregate_results(
    datasets=['german', 'bail', 'pokec_z', 'pokec_n', 'nba'],
    encoders=['gcn', 'gat', 'gin', 'sage', 'sgc'],
    models=['edge_adder'],
    root=root,
    verbose=True
)
df.shape

(250, 13)

In [ ]:
df.to_csv('ea_non-mm/ea_dp_or_eo_only.csv', index=False)

In [11]:
c1 = df["dataset"] == "bail"
c2 = df["backbone"] == "gcn"
df[c1 & c2]

,dataset,backbone,model,lambda_dp,lambda_eo,AUC_mean,AUC_std,F1_mean,F1_std,DP_mean,DP_std,EO_mean,EO_std
55,bail,gcn,edge_adder,0.0,0.1,0.899686,0.001060,0.814338,0.006320,0.079437,0.005103,0.047908,0.002679
105,bail,gcn,edge_adder,0.0,0.5,0.903677,0.001196,0.814697,0.008266,0.077456,0.002199,0.046925,0.002508
5,bail,gcn,edge_adder,0.0,1.0,0.906182,0.000722,0.823771,0.006899,0.073847,0.001090,0.045550,0.003088
80,bail,gcn,edge_adder,0.0,5.0,0.912169,0.003030,0.832004,0.005512,0.049806,0.004960,0.030902,0.006870
30,bail,gcn,edge_adder,0.0,10.0,0.909127,0.003507,0.830681,0.006621,0.039144,0.005058,0.021785,0.004359
180,bail,gcn,edge_adder,0.1,0.0,0.897838,0.001284,0.812525,0.005426,0.079646,0.004410,0.049260,0.002294
230,bail,gcn,edge_adder,0.5,0.0,0.896280,0.001914,0.811033,0.008049,0.077971,0.004329,0.047313,0.003902
155,bail,gcn,edge_adder,1.0,0.0,0.896329,0.000971,0.806542,0.004967,0.079968,0.003027,0.047584,0.003830
205,bail,gcn,edge_adder,5.0,0.0,0.887059,0.001350,0.799062,0.004145,0.076512,0.002266,0.044546,0.001728
130,bail,gcn,edge_adder,10.0,0.0,0.881704,0.001065,0.787510,0.005320,0.075919,0.001857,0.041836,0.002130


## Concatenate with vanilla results

In [12]:
import json

In [19]:
ROOT = "./tuned_results/vanilla"
files = os.listdir(ROOT)
lst_data = []
for file in files:
    if not file.endswith('.json'):
        continue
    path = os.path.join(ROOT, file)
    with open(path, 'r') as f:
        data = json.load(f)
    reformatted_data = {
        "dataset": file.split("__")[0],
        "backbone": file.split('__')[1].split('_')[0],
        "model": "vanilla",
        "lambda_dp": 0.0,
        "lambda_eo": 0.0,
        "AUC_mean": data["auc"]["mean"],
        "AUC_std": data["auc"]["std"],
        "F1_mean": data["f1"]["mean"],
        "F1_std": data["f1"]["std"],
        "DP_mean": data["dp"]["mean"],
        "DP_std": data["dp"]["std"],
        "EO_mean": data["eo"]["mean"],
        "EO_std": data["eo"]["std"],
    }
    lst_data.append(reformatted_data)

In [24]:
df_vanilla = pd.DataFrame(lst_data)
df_concatenated = pd.concat([df, df_vanilla], ignore_index=True)
df_concatenated.sort_values(by=['dataset', 'backbone', 'lambda_dp', 'lambda_eo'], inplace=True)

In [25]:
c1 = df_concatenated["dataset"] == "bail"
c2 = df_concatenated["backbone"] == "gcn"
df_concatenated[c1 & c2]

,dataset,backbone,model,lambda_dp,lambda_eo,AUC_mean,AUC_std,F1_mean,F1_std,DP_mean,DP_std,EO_mean,EO_std
251,bail,gcn,vanilla,0.0,0.0,0.902179,0.001961,0.805811,0.009484,0.061524,0.001703,0.032869,0.004186
10,bail,gcn,edge_adder,0.0,0.1,0.899686,0.001060,0.814338,0.006320,0.079437,0.005103,0.047908,0.002679
11,bail,gcn,edge_adder,0.0,0.5,0.903677,0.001196,0.814697,0.008266,0.077456,0.002199,0.046925,0.002508
12,bail,gcn,edge_adder,0.0,1.0,0.906182,0.000722,0.823771,0.006899,0.073847,0.001090,0.045550,0.003088
13,bail,gcn,edge_adder,0.0,5.0,0.912169,0.003030,0.832004,0.005512,0.049806,0.004960,0.030902,0.006870
14,bail,gcn,edge_adder,0.0,10.0,0.909127,0.003507,0.830681,0.006621,0.039144,0.005058,0.021785,0.004359
15,bail,gcn,edge_adder,0.1,0.0,0.897838,0.001284,0.812525,0.005426,0.079646,0.004410,0.049260,0.002294
16,bail,gcn,edge_adder,0.5,0.0,0.896280,0.001914,0.811033,0.008049,0.077971,0.004329,0.047313,0.003902
17,bail,gcn,edge_adder,1.0,0.0,0.896329,0.000971,0.806542,0.004967,0.079968,0.003027,0.047584,0.003830
18,bail,gcn,edge_adder,5.0,0.0,0.887059,0.001350,0.799062,0.004145,0.076512,0.002266,0.044546,0.001728


In [26]:
df_concatenated.to_csv('ea_non-mm/ea_dp_or_eo_only_with_vanilla.csv', index=False)

# After 18 Oct 2025 (`ea_minmax`)

## Use edge-adder-tuned HPs with default min-max HPs

In [17]:
Results = {}
root = '../logs/runs_edge_minmax'

In [25]:
for dataset in ['german', 'bail', 'pokec_z', 'pokec_n', 'nba']:
    for encoder in ['gcn', 'gat', 'gin', 'sage', 'sgc']:
        for model in ['edge_minmax']:
            try:
                results = get_result_json(dataset, encoder, model, root)
                Results[f'{dataset}_{encoder}_{model}'] = results
                print(f'{dataset}_{encoder}_{model}:')
                for metrics in ['auc', 'f1', 'dp', 'eo']:
                    m = results['results'][f'{metrics.upper()}_mean']
                    s = results['results'][f'{metrics.upper()}_std']
                    print(f'{m:.2%} ± {s:.2%}')
            except Exception as e:
                print(f'Error processing {dataset}_{encoder}_{model}: {e}')
            print()


Found 1 log files in ../logs/runs_edge_minmax/german/gcn/edge_minmax/20251019-033815
german_gcn_edge_minmax:
74.44% ± 0.88%
67.91% ± 17.19%
32.03% ± 7.50%
25.78% ± 3.17%

Log directory does not exist: ../logs/runs_edge_minmax/german/gat/edge_minmax
german_gat_edge_minmax:
Error processing german_gat_edge_minmax: 'NoneType' object is not subscriptable

Found 1 log files in ../logs/runs_edge_minmax/german/gin/edge_minmax/20251019-041022
german_gin_edge_minmax:
68.77% ± 2.08%
74.16% ± 8.48%
20.60% ± 11.98%
18.60% ± 12.54%

Found 1 log files in ../logs/runs_edge_minmax/german/sage/edge_minmax/20251019-041440
german_sage_edge_minmax:
68.92% ± 4.90%
58.96% ± 20.16%
23.51% ± 11.14%
23.16% ± 11.58%

Found 1 log files in ../logs/runs_edge_minmax/german/sgc/edge_minmax/20251019-041114
german_sgc_edge_minmax:
70.18% ± 0.90%
66.19% ± 9.84%
28.09% ± 8.24%
22.50% ± 7.27%

Found 1 log files in ../logs/runs_edge_minmax/bail/gcn/edge_minmax/20251018-213200
bail_gcn_edge_minmax:
88.97% ± 0.56%
81.05% ± 

## Use default HPs

In [5]:
Results = {}
root = '../logs/runs/edge_minmax'

In [7]:
for dataset in ['german', 'bail', 'pokec_z', 'pokec_n', 'nba']:
    for encoder in ['gcn', 'gat']:
        for model in ['edge_minmax']:
            results = get_result_json(dataset, encoder, model, root)
            Results[f'{dataset}_{encoder}_{model}'] = results
            print(f'{dataset}_{encoder}_{model}:')
            for metrics in ['auc', 'f1', 'dp', 'eo']:
                m = results['results'][f'{metrics.upper()}_mean']
                s = results['results'][f'{metrics.upper()}_std']
                print(f'{m:.2%} ± {s:.2%}')

Found 1 log files in ../logs/runs/edge_minmax/german/gcn/edge_minmax/20251018-102209
german_gcn_edge_minmax:
67.39% ± 1.36%
59.67% ± 21.86%
24.12% ± 9.25%
19.89% ± 6.17%
Found 1 log files in ../logs/runs/edge_minmax/german/gat/edge_minmax/20251018-102341
german_gat_edge_minmax:
53.72% ± 7.61%
32.30% ± 39.59%
2.51% ± 7.31%
2.47% ± 7.13%
Found 1 log files in ../logs/runs/edge_minmax/bail/gcn/edge_minmax/20251018-103043
bail_gcn_edge_minmax:
85.81% ± 0.64%
74.78% ± 1.68%
7.69% ± 1.07%
4.36% ± 1.27%
Found 1 log files in ../logs/runs/edge_minmax/bail/gat/edge_minmax/20251018-103321
bail_gat_edge_minmax:
89.11% ± 0.75%
78.61% ± 1.30%
6.75% ± 0.76%
4.52% ± 1.18%
Found 1 log files in ../logs/runs/edge_minmax/pokec_z/gcn/edge_minmax/20251018-105147
pokec_z_gcn_edge_minmax:
77.99% ± 0.25%
72.05% ± 0.74%
5.17% ± 1.08%
3.86% ± 1.02%
Found 1 log files in ../logs/runs/edge_minmax/pokec_z/gat/edge_minmax/20251018-105244
pokec_z_gat_edge_minmax:
77.15% ± 0.83%
68.79% ± 2.14%
3.61% ± 1.62%
2.92% ± 1.69

In [ ]:
with open('all_results_0-9.json', 'w') as f:
    json.dump(Results, f, indent=4)

# Before 18 Oct 2025

## Seeds 42~51

In [ ]:
ROOT = '../logs/run/edge_adder'
Results = {}
for dataset in ['german', 'bail', 'pokec_z', 'pokec_n', 'nba']:
    for encoder in ['gcn', 'gat', 'gin', 'sage', 'sgc']:
        print(f"\nProcessing {dataset} with {encoder}")
        for model in ['edge_adder']:
            results = get_result_json(dataset, encoder, model, root=ROOT)
            Results[f'{dataset}_{encoder}_{model}'] = results
            print(f'{dataset}_{encoder}_{model}:')
            for metrics in ['auc', 'f1', 'dp', 'eo']:
                try:
                    m = results['results'][f'{metrics.upper()}_mean']
                    s = results['results'][f'{metrics.upper()}_std']
                except KeyError:
                    m = results['results']['SP_mean']
                    s = results['results']['SP_std']
                print(f'{m:.2%} ± {s:.2%}')


Processing german with gcn
Found 1 log files in ../logs/run/edge_adder/german/gcn/edge_adder/20250927-233744
german_gcn_edge_adder:
71.22% ± 1.98%
51.86% ± 22.07%
7.15% ± 4.20%
3.82% ± 2.99%

Processing german with gat
Found 1 log files in ../logs/run/edge_adder/german/gat/edge_adder/20250927-235613
german_gat_edge_adder:
53.14% ± 5.69%
41.12% ± 41.12%
0.13% ± 0.38%
0.17% ± 0.50%

Processing german with gin
Found 1 log files in ../logs/run/edge_adder/german/gin/edge_adder/20250927-235727
german_gin_edge_adder:
59.65% ± 8.43%
72.61% ± 16.72%
10.43% ± 10.89%
8.71% ± 10.63%

Processing german with sage
Found 1 log files in ../logs/run/edge_adder/german/sage/edge_adder/20250927-235850
german_sage_edge_adder:
61.18% ± 6.02%
38.01% ± 26.38%
15.12% ± 12.51%
15.53% ± 13.42%

Processing german with sgc
Found 1 log files in ../logs/run/edge_adder/german/sgc/edge_adder/20250927-234656
german_sgc_edge_adder:
68.10% ± 1.43%
40.21% ± 24.37%
6.23% ± 4.99%
4.23% ± 3.92%

Processing bail with gcn
Foun

In [ ]:
ROOT = '../logs/run/edge_adder'
Results = {}
for dataset in ['german', 'bail', 'pokec_z', 'pokec_n', 'nba']:
    for encoder in ['gcn']:
        for model in ['edge_adder']:
            results = get_result_json(dataset, encoder, model, root=ROOT)
            Results[f'{dataset}_{encoder}_{model}'] = results
            print(f'{dataset}_{encoder}_{model}:')
            for metrics in ['auc', 'f1', 'dp', 'eo']:
                try:
                    m = results['results'][f'{metrics.upper()}_mean']
                    s = results['results'][f'{metrics.upper()}_std']
                except KeyError:
                    m = results['results']['SP_mean']
                    s = results['results']['SP_std']
                print(f'{m:.2%} ± {s:.2%}')

In [24]:
import pickle as pkl

In [ ]:
with open('all_results_42-51.json', 'w') as f:
    json.dump(Results, f, indent=4)

## Seed 0~9

In [33]:
Results = {}
for dataset in ['german', 'bail', 'pokec_z', 'pokec_n', 'nba']:
    for encoder in ['gcn']:
        for model in ['vanilla', 'fairinv']:
            results = get_result_json(dataset, encoder, model)
            Results[f'{dataset}_{encoder}_{model}'] = results
            print(f'{dataset}_{encoder}_{model}:')
            for metrics in ['auc', 'f1', 'dp', 'eo']:
                m = results['results'][f'{metrics.upper()}_mean']
                s = results['results'][f'{metrics.upper()}_std']
                print(f'{m:.2%} ± {s:.2%}')

Found 1 log files in ./logs/german/gcn/vanilla/20250926-204322
german_gcn_vanilla:
70.68% ± 4.25%
62.53% ± 20.52%
12.14% ± 11.15%
10.59% ± 9.34%
Found 1 log files in ./logs/german/gcn/fairinv/20250926-204139
german_gcn_fairinv:
68.63% ± 1.35%
68.94% ± 26.71%
1.33% ± 1.84%
1.13% ± 1.62%
Found 1 log files in ./logs/bail/gcn/vanilla/20250926-204508
bail_gcn_vanilla:
90.61% ± 0.05%
80.56% ± 0.12%
6.44% ± 0.08%
3.23% ± 0.16%
Found 1 log files in ./logs/bail/gcn/fairinv/20250926-204535
bail_gcn_fairinv:
89.29% ± 1.42%
79.29% ± 2.35%
4.19% ± 1.25%
2.53% ± 0.95%
Found 1 log files in ./logs/pokec_z/gcn/vanilla/20250926-204752
pokec_z_gcn_vanilla:
75.06% ± 0.10%
68.79% ± 1.46%
4.71% ± 0.34%
4.77% ± 0.50%
Found 1 log files in ./logs/pokec_z/gcn/fairinv/20250926-205159
pokec_z_gcn_fairinv:
75.29% ± 0.84%
69.87% ± 0.56%
4.35% ± 1.05%
3.58% ± 1.11%
Found 1 log files in ./logs/pokec_n/gcn/vanilla/20250926-205029
pokec_n_gcn_vanilla:
72.72% ± 0.06%
64.04% ± 0.67%
0.66% ± 0.25%
2.07% ± 0.85%
Found 1 lo

In [ ]:
with open('all_results_0-9.json', 'w') as f:
    json.dump(Results, f, indent=4)

## Seeds 0 ~ 4 (5 runs)

In [35]:
Results = {}
for dataset in ['german', 'bail', 'pokec_z', 'pokec_n', 'nba']:
    for encoder in ['gcn']:
        for model in ['vanilla', 'fairinv']:
            results = get_result_json(dataset, encoder, model)
            Results[f'{dataset}_{encoder}_{model}'] = results
            print(f'{dataset}_{encoder}_{model}:')
            for metrics in ['auc', 'f1', 'dp', 'eo']:
                m = results['results'][f'{metrics.upper()}_mean']
                s = results['results'][f'{metrics.upper()}_std']
                print(f'{m:.2%} ± {s:.2%}')

Found 1 log files in ./logs/german/gcn/vanilla/20250926-211351
german_gcn_vanilla:
70.06% ± 4.00%
68.91% ± 16.21%
8.78% ± 9.17%
8.30% ± 8.23%
Found 1 log files in ./logs/german/gcn/fairinv/20250926-211500
german_gcn_fairinv:
68.30% ± 0.99%
82.26% ± 0.11%
0.72% ± 1.00%
0.55% ± 0.72%
Found 1 log files in ./logs/bail/gcn/vanilla/20250926-211422
bail_gcn_vanilla:
90.61% ± 0.04%
80.59% ± 0.10%
6.46% ± 0.05%
3.22% ± 0.15%
Found 1 log files in ./logs/bail/gcn/fairinv/20250926-211705
bail_gcn_fairinv:
88.59% ± 0.75%
78.36% ± 1.85%
4.77% ± 0.79%
2.67% ± 0.74%
Found 1 log files in ./logs/pokec_z/gcn/vanilla/20250926-211516
pokec_z_gcn_vanilla:
75.03% ± 0.14%
68.40% ± 1.97%
4.61% ± 0.40%
4.74% ± 0.65%
Found 1 log files in ./logs/pokec_z/gcn/fairinv/20250926-211949
pokec_z_gcn_fairinv:
75.60% ± 0.53%
70.02% ± 0.59%
3.83% ± 0.96%
3.33% ± 1.14%
Found 1 log files in ./logs/pokec_n/gcn/vanilla/20250926-211628
pokec_n_gcn_vanilla:
72.72% ± 0.07%
63.77% ± 0.59%
0.75% ± 0.22%
2.38% ± 0.80%
Found 1 log fi

In [36]:
with open('all_results_0-4.json', 'w') as f:
    json.dump(Results, f, indent=4)

In [48]:
Results = {}
for dataset in ['german', 'bail', 'pokec_z', 'pokec_n', 'nba']:
    Results[dataset] = {}
    for encoder in ['gat']:
        Results[dataset][encoder] = {}
        for model in ['vanilla', 'fairinv']:
            results = get_result_json(dataset, encoder, model)
            Results[dataset][encoder][model] = results
            print(f'{dataset}_{encoder}_{model}:')
            for metrics in ['auc', 'f1', 'dp', 'eo']:
                try:
                    m = results['results'][f'{metrics.upper()}_mean']
                    s = results['results'][f'{metrics.upper()}_std']
                except KeyError:
                    m = results['results']['SP_mean']
                    s = results['results']['SP_std']
                print(f'{m:.2%} ± {s:.2%}')

Found 1 log files in ./logs/german/gat/vanilla/20250927-113344
german_gat_vanilla:
52.75% ± 5.84%
16.47% ± 32.94%
0.00% ± 0.00%
0.00% ± 0.00%
Found 1 log files in ./logs/german/gat/fairinv/20250926-185751
german_gat_fairinv:
53.76% ± 3.74%
16.47% ± 32.94%
0.00% ± 0.00%
0.00% ± 0.00%
Found 1 log files in ./logs/bail/gat/vanilla/20250927-113728
bail_gat_vanilla:
93.17% ± 0.90%
84.60% ± 1.81%
7.50% ± 1.06%
4.23% ± 0.96%
Found 1 log files in ./logs/bail/gat/fairinv/20250926-190210
bail_gat_fairinv:
93.20% ± 1.04%
77.09% ± 9.07%
6.04% ± 1.04%
3.44% ± 1.91%
Found 1 log files in ./logs/pokec_z/gat/vanilla/20250927-114212
pokec_z_gat_vanilla:
76.13% ± 0.60%
70.00% ± 0.71%
4.71% ± 1.48%
4.48% ± 1.76%
Found 1 log files in ./logs/pokec_z/gat/fairinv/20250926-191027
pokec_z_gat_fairinv:
77.35% ± 0.71%
69.45% ± 2.95%
3.90% ± 1.93%
3.68% ± 2.10%
Found 1 log files in ./logs/pokec_n/gat/vanilla/20250927-114729
pokec_n_gat_vanilla:
73.71% ± 0.44%
65.80% ± 1.94%
1.53% ± 1.22%
1.82% ± 1.27%
Found 1 log f

In [49]:
for dataset in ['german', 'bail', 'pokec_z', 'pokec_n', 'nba']:
    for encoder in ['gin']:
        Results[dataset][encoder] = {}
        for model in ['vanilla', 'fairinv']:
            results = get_result_json(dataset, encoder, model)
            Results[dataset][encoder][model] = results
            print(f'{dataset}_{encoder}_{model}:')
            for metrics in ['auc', 'f1', 'dp', 'eo']:
                try:
                    m = results['results'][f'{metrics.upper()}_mean']
                    s = results['results'][f'{metrics.upper()}_std']
                except KeyError:
                    m = results['results']['SP_mean']
                    s = results['results']['SP_std']
                print(f'{m:.2%} ± {s:.2%}')

Found 1 log files in ./logs/german/gin/vanilla/20250926-195655
german_gin_vanilla:
55.89% ± 6.55%
62.04% ± 30.20%
9.17% ± 15.22%
9.50% ± 15.00%
Found 1 log files in ./logs/german/gin/fairinv/20250926-200251
german_gin_fairinv:
68.17% ± 2.64%
57.70% ± 37.77%
0.36% ± 0.56%
0.08% ± 0.25%
Found 1 log files in ./logs/bail/gin/vanilla/20250926-195745
bail_gin_vanilla:
84.45% ± 1.06%
72.52% ± 2.27%
7.88% ± 1.78%
4.94% ± 1.40%
Found 1 log files in ./logs/bail/gin/fairinv/20250926-200507
bail_gin_fairinv:
84.23% ± 1.43%
72.99% ± 2.24%
7.52% ± 2.21%
5.04% ± 1.72%
Found 1 log files in ./logs/pokec_z/gin/vanilla/20250926-195905
pokec_z_gin_vanilla:
70.93% ± 0.80%
66.58% ± 2.38%
5.17% ± 1.94%
5.31% ± 3.13%
Found 1 log files in ./logs/pokec_z/gin/fairinv/20250926-200848
pokec_z_gin_fairinv:
75.15% ± 0.34%
70.32% ± 1.20%
2.62% ± 2.38%
2.45% ± 2.03%
Found 1 log files in ./logs/pokec_n/gin/vanilla/20250926-200034
pokec_n_gin_vanilla:
69.63% ± 1.54%
61.06% ± 2.86%
2.10% ± 1.44%
2.48% ± 2.45%
Found 1 log

In [53]:
for dataset in ['german', 'bail', 'pokec_z', 'pokec_n', 'nba']:
    for encoder in ['sage']:
        Results[dataset][encoder] = {}
        for model in ['vanilla', 'fairinv']:
            results = get_result_json(dataset, encoder, model)
            Results[dataset][encoder][model] = results
            print(f'{dataset}_{encoder}_{model}:')
            for metrics in ['auc', 'f1', 'dp', 'eo']:
                try:
                    m = results['results'][f'{metrics.upper()}_mean']
                    s = results['results'][f'{metrics.upper()}_std']
                except KeyError:
                    m = results['results']['SP_mean']
                    s = results['results']['SP_std']
                print(f'{m:.2%} ± {s:.2%}')

Found 1 log files in ./logs/german/sage/vanilla/20250927-095950
german_sage_vanilla:
67.82% ± 3.20%
59.65% ± 23.25%
35.04% ± 14.14%
34.58% ± 14.54%
Found 1 log files in ./logs/german/sage/fairinv/20250927-112655
german_sage_fairinv:
69.74% ± 3.55%
82.21% ± 0.46%
1.76% ± 2.43%
0.72% ± 0.94%
Found 1 log files in ./logs/bail/sage/vanilla/20250927-100049
bail_sage_vanilla:
91.72% ± 0.63%
82.38% ± 1.93%
2.75% ± 2.52%
1.46% ± 1.29%
Found 1 log files in ./logs/bail/sage/fairinv/20250927-113014
bail_sage_fairinv:
91.03% ± 0.96%
81.17% ± 1.55%
1.17% ± 0.89%
1.28% ± 0.71%
Found 1 log files in ./logs/pokec_z/sage/vanilla/20250927-100225
pokec_z_sage_vanilla:
72.12% ± 1.14%
69.22% ± 0.89%
3.00% ± 1.48%
3.76% ± 1.91%
Found 1 log files in ./logs/pokec_z/sage/fairinv/20250927-113418
pokec_z_sage_fairinv:
78.12% ± 0.48%
71.14% ± 1.32%
5.10% ± 2.44%
4.60% ± 2.38%
Found 1 log files in ./logs/pokec_n/sage/vanilla/20250927-100423
pokec_n_sage_vanilla:
68.28% ± 1.32%
64.04% ± 1.43%
1.04% ± 0.87%
1.60% ± 1.

In [55]:
for dataset in ['german', 'bail', 'pokec_z', 'pokec_n', 'nba']:
    for encoder in ['sgc']:
        Results[dataset][encoder] = {}
        for model in ['vanilla', 'fairinv']:
            results = get_result_json(dataset, encoder, model)
            Results[dataset][encoder][model] = results
            print(f'{dataset}_{encoder}_{model}:')
            for metrics in ['auc', 'f1', 'dp', 'eo']:
                try:
                    m = results['results'][f'{metrics.upper()}_mean']
                    s = results['results'][f'{metrics.upper()}_std']
                except KeyError:
                    m = results['results']['SP_mean']
                    s = results['results']['SP_std']
                print(f'{m:.2%} ± {s:.2%}')

Found 1 log files in ./logs/german/sgc/vanilla/20250926-200446
german_sgc_vanilla:
68.45% ± 1.84%
55.93% ± 28.37%
13.55% ± 10.12%
11.46% ± 7.34%
Found 1 log files in ./logs/german/sgc/fairinv/20250927-122013
german_sgc_fairinv:
65.79% ± 1.28%
74.46% ± 23.33%
1.04% ± 1.26%
1.14% ± 1.39%
Found 1 log files in ./logs/bail/sgc/vanilla/20250926-200621
bail_sgc_vanilla:
88.24% ± 0.04%
78.01% ± 0.28%
4.72% ± 0.18%
3.40% ± 0.15%
Found 1 log files in ./logs/bail/sgc/fairinv/20250927-122330
bail_sgc_fairinv:
87.50% ± 0.75%
75.05% ± 2.74%
3.94% ± 1.37%
1.83% ± 0.94%
Found 1 log files in ./logs/pokec_z/sgc/vanilla/20250926-200847
pokec_z_sgc_vanilla:
76.90% ± 0.03%
70.65% ± 0.68%
3.09% ± 0.38%
3.22% ± 0.62%
Found 1 log files in ./logs/pokec_z/sgc/fairinv/20250927-122800
pokec_z_sgc_fairinv:
76.12% ± 1.19%
71.37% ± 0.50%
3.11% ± 1.80%
2.11% ± 1.72%
Found 1 log files in ./logs/pokec_n/sgc/vanilla/20250926-201229
pokec_n_sgc_vanilla:
75.11% ± 0.05%
67.04% ± 0.81%
1.46% ± 0.47%
1.47% ± 0.40%
Found 1 lo

In [60]:
with open('all_results_42-51.json', 'r') as f:
    Results_42_51 = json.load(f)
Results_42_51.keys()

dict_keys(['german_gcn_vanilla', 'german_gcn_fairinv', 'bail_gcn_vanilla', 'bail_gcn_fairinv', 'pokec_z_gcn_vanilla', 'pokec_z_gcn_fairinv', 'pokec_n_gcn_vanilla', 'pokec_n_gcn_fairinv', 'nba_gcn_vanilla', 'nba_gcn_fairinv'])

In [62]:
for k in Results_42_51.keys():
    k_split = k.split('_')
    dataset, encoder, model = k_split[0], k_split[1], k_split[2]
    if encoder != 'gcn':
        continue
    if encoder not in Results[dataset]:
        Results[dataset][encoder] = {}
    Results[dataset][encoder][model] = Results_42_51[k]

In [65]:
Results['german']['gcn']['vanilla']['results']

{'AUC_mean': 0.7161828571428572,
 'AUC_std': 0.030847160290036143,
 'F1_mean': 0.47361186680866013,
 'F1_std': 0.16334220857187437,
 'ACC_mean': 0.506,
 'ACC_std': 0.08679631328576115,
 'SP_mean': 0.14686306601200216,
 'SP_std': 0.07126286684567669,
 'EO_mean': 0.12195378151260505,
 'EO_std': 0.05854045408513699,
 'name': 'vanilla_german_gcn_alpha:10.0_lr_sp:0.1_env_num:2_lr:0.01'}

In [68]:
with open('all_encoder_results_42-51.json', 'w') as f:
    json.dump(Results, f, indent=4)

In [75]:
all_encoder_results_0_4 = {}
for dataset in ['german', 'bail', 'pokec_z', 'pokec_n']:
    all_encoder_results_0_4[dataset] = {}
    for encoder in ['gin']:
        all_encoder_results_0_4[dataset][encoder] = {}
        for model in ['vanilla', 'fairinv']:
            results = get_result_json(dataset, encoder, model)
            all_encoder_results_0_4[dataset][encoder][model] = results
            print(f'{dataset}_{encoder}_{model}:')
            for metrics in ['auc', 'f1', 'dp', 'eo']:
                try:
                    m = results['results'][f'{metrics.upper()}_mean']
                    s = results['results'][f'{metrics.upper()}_std']
                except KeyError:
                    m = results['results']['SP_mean']
                    s = results['results']['SP_std']
                print(f'{m:.2%} ± {s:.2%}')

Found 1 log files in ./logs/german/gin/vanilla/20250927-132259
german_gin_vanilla:
57.67% ± 10.00%
68.92% ± 23.95%
11.23% ± 11.10%
10.36% ± 10.81%
Found 1 log files in ./logs/german/gin/fairinv/20250927-132344
german_gin_fairinv:
63.84% ± 3.06%
33.21% ± 40.21%
0.34% ± 0.44%
0.17% ± 0.34%
Found 1 log files in ./logs/bail/gin/vanilla/20250927-132326
bail_gin_vanilla:
84.36% ± 0.84%
72.52% ± 2.15%
8.95% ± 1.64%
5.56% ± 1.85%
Found 1 log files in ./logs/bail/gin/fairinv/20250927-132515
bail_gin_fairinv:
84.11% ± 1.05%
72.60% ± 2.17%
7.90% ± 2.11%
5.71% ± 1.43%
Found 1 log files in ./logs/pokec_z/gin/vanilla/20250927-132415
pokec_z_gin_vanilla:
71.57% ± 1.09%
65.01% ± 3.37%
4.50% ± 1.11%
5.73% ± 1.07%
Found 1 log files in ./logs/pokec_z/gin/fairinv/20250927-132700
pokec_z_gin_fairinv:
73.84% ± 1.61%
69.66% ± 1.34%
1.69% ± 0.90%
0.83% ± 0.65%
Found 1 log files in ./logs/pokec_n/gin/vanilla/20250927-132515
pokec_n_gin_vanilla:
70.81% ± 0.56%
62.16% ± 1.06%
0.94% ± 0.53%
1.25% ± 0.86%
Found 1 

In [76]:
for dataset in ['german', 'bail', 'pokec_z', 'pokec_n']:
    for encoder in ['sage']:
        all_encoder_results_0_4[dataset][encoder] = {}
        for model in ['vanilla', 'fairinv']:
            results = get_result_json(dataset, encoder, model)
            all_encoder_results_0_4[dataset][encoder][model] = results
            print(f'{dataset}_{encoder}_{model}:')
            for metrics in ['auc', 'f1', 'dp', 'eo']:
                try:
                    m = results['results'][f'{metrics.upper()}_mean']
                    s = results['results'][f'{metrics.upper()}_std']
                except KeyError:
                    m = results['results']['SP_mean']
                    s = results['results']['SP_std']
                print(f'{m:.2%} ± {s:.2%}')

Found 1 log files in ./logs/german/sage/vanilla/20250927-132809
german_sage_vanilla:
66.06% ± 2.48%
52.39% ± 15.79%
27.73% ± 6.96%
29.14% ± 6.36%
Found 1 log files in ./logs/german/sage/fairinv/20250927-134114
german_sage_fairinv:
72.29% ± 4.19%
69.74% ± 22.88%
7.17% ± 4.59%
4.45% ± 2.25%
Found 1 log files in ./logs/bail/sage/vanilla/20250927-132904
bail_sage_vanilla:
91.42% ± 0.28%
81.24% ± 2.84%
1.77% ± 3.13%
1.13% ± 0.61%
Found 1 log files in ./logs/bail/sage/fairinv/20250927-134241
bail_sage_fairinv:
91.43% ± 0.69%
81.15% ± 1.44%
0.64% ± 0.64%
1.14% ± 1.06%
Found 1 log files in ./logs/pokec_z/sage/vanilla/20250927-133015
pokec_z_sage_vanilla:
72.97% ± 0.93%
69.05% ± 1.82%
3.22% ± 1.46%
2.21% ± 1.44%
Found 1 log files in ./logs/pokec_z/sage/fairinv/20250927-134443
pokec_z_sage_fairinv:
77.70% ± 1.29%
71.72% ± 1.06%
4.57% ± 3.13%
4.57% ± 2.59%
Found 1 log files in ./logs/pokec_n/sage/vanilla/20250927-133130
pokec_n_sage_vanilla:
70.77% ± 0.59%
64.98% ± 0.64%
1.65% ± 1.14%
0.87% ± 0.4

In [77]:
with open('gin_sage_results_0_4.json', 'w') as f:
    json.dump(all_encoder_results_0_4, f, indent=4)